# Iteration 4: Enhanced Multi-Model Pipeline with Advanced Techniques — RQ1
This iteration extends the multilingual embedding comparison from Iteration 3 by introducing **advanced machine learning techniques** designed to improve Process Safety classification performance. Ten embedding models are evaluated across four country-specific datasets using five distinct classifiers, incorporating SMOTE oversampling, mean pooling, hyperparameter tuning, and ensemble methods.

## Experimental Design

| Component | Configuration |
|---|---|
| **Embedding Models** | bert-base-multilingual-uncased, xlm-roberta-base, paraphrase-multilingual-mpnet-base-v2, paraphrase-multilingual-MiniLM-L12-v2, snowflake-arctic-embed-l-v2.0, labse-gguf, bge-m3-korean, gte-multilingual-base, multilingual-e5-large-instruct, e5-base-multilingual-4096 |
| **Classifiers** | LinearSVC (GridSearchCV-tuned), Random Forest, XGBoost, LightGBM, Voting Ensemble |
| **Pooling Strategy** | Mean pooling with attention mask |
| **Class Balancing** | SMOTE (minority oversampled to 70% of majority) |
| **Feature Scaling** | StandardScaler (fitted on original training split) |
| **Datasets** | English, Germany, Sweden, Netherlands (manual country-level splits) |

## Key Enhancements over Iteration 3

1. **SMOTE oversampling** to address class imbalance in the minority (Process Safety) class
2. **Mean pooling** with attention mask instead of CLS-token pooling for richer sentence representations
3. **Multiple classifiers** (SVM, Random Forest, XGBoost, LightGBM) compared systematically
4. **Hyperparameter tuning** via GridSearchCV for LinearSVC
5. **Voting Ensemble** combining top-3 classifiers by Process Safety F1

## Outputs

- Cached embeddings in `Embeddings/_iteration_4/{model_name}/`
- Per-classifier confusion matrices, ROC curves, and precision-recall curves (PNG + PDF, 300 dpi)
- Comprehensive results CSV and summary JSON in `Results/_iteration_4/`

In [1]:
# =============================================================================
# CONFIGURATION & RESET UTILITIES — ITERATION 4
# =============================================================================
# Cross-platform path resolution (consistent with Iteration 0).
# Uses find_project_root_with_datasets() to locate project root automatically.

import os
import json
import glob
import shutil
from pathlib import Path

def find_project_root_with_datasets(start_path: Path, max_levels: int = 10) -> Path:
    """Search upward from start_path for directory containing 'Datasets' folder."""
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / 'Datasets').exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not find project root with 'Datasets' folder within {max_levels} levels. "
        f"Set THESIS_BASE_DIR environment variable or ensure Datasets folder exists."
    )

env_base = os.environ.get('THESIS_BASE_DIR')
if env_base:
    BASE_DIR = Path(env_base)
    print(f"Using THESIS_BASE_DIR from environment: {BASE_DIR}")
else:
    BASE_DIR = find_project_root_with_datasets(Path.cwd())
    print(f"Found project root: {BASE_DIR}")

PATHS = {
    'master_dataset': BASE_DIR / "Master Dataset 34k",
    'embeddings':     BASE_DIR / "Embeddings" / "_iteration_4",
    'results':        BASE_DIR / "Results" / "_iteration_4"
}

for k, p in list(PATHS.items()):
    PATHS[k] = Path(p).resolve()
    PATHS[k].mkdir(parents=True, exist_ok=True)

DATA_DIR = str(PATHS['master_dataset'])
EMBEDDINGS_BASE_DIR = str(PATHS['embeddings'])
RESULTS_DIR = str(PATHS['results'])
CHECKPOINT_FILE = str(PATHS['results'] / "checkpoint_iteration_4.json")  # or "checkpoint.json" for consistency

print(f"BASE_DIR:            {BASE_DIR}")
print(f"DATA_DIR:            {DATA_DIR}")
print(f"EMBEDDINGS_BASE_DIR: {EMBEDDINGS_BASE_DIR}")
print(f"RESULTS_DIR:         {RESULTS_DIR}")
print(f"CHECKPOINT_FILE:     {CHECKPOINT_FILE}")

# Language code mapping (consistent with Iteration 0)
LANGUAGE_CODE_MAP = {
    'English': 'EN', 'German': 'DE', 'Swedish': 'SV',
    'Dutch':   'NL', 'Hungarian': 'HU', 'Unknown': 'UN',
}


def reset_checkpoint():
    """Delete checkpoint to restart from beginning"""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print(" Checkpoint cleared - will restart from beginning")
    else:
        print(" No checkpoint found - already clean")

def view_progress():
    """View current progress"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
            processed = data['processed']
            print(f"\n{'='*60}")
            print(f"ITERATION 4 PROGRESS: {len(processed)} combinations processed")
            print(f"{'='*60}")
            print("\nCompleted:")
            for item in processed:
                if isinstance(item, (list, tuple)) and len(item) >= 2:
                    model, dataset = item[0], item[1]
                    print(f"   {model:45s} → {dataset}")
                else:
                    print(f"   {item}")
    else:
        print(" No checkpoint found - no progress yet")

def reset_embeddings():
    """Delete all generated embeddings for iteration 4"""
    if os.path.exists(EMBEDDINGS_BASE_DIR):
        response = input(f" Delete ALL embeddings in {EMBEDDINGS_BASE_DIR}? (yes/no): ")
        if response.lower() == 'yes':
            shutil.rmtree(EMBEDDINGS_BASE_DIR)
            os.makedirs(EMBEDDINGS_BASE_DIR)
            print(" All iteration 4 embeddings deleted")
        else:
            print(" Cancelled")
    else:
        print(" Embeddings directory doesn't exist")

def reset_results():
    """Delete all results for iteration 4"""
    if os.path.exists(RESULTS_DIR):
        response = input(f" Delete ALL results in {RESULTS_DIR}? (yes/no): ")
        if response.lower() == 'yes':
            for file in glob.glob(os.path.join(RESULTS_DIR, '*')):
                if os.path.isfile(file):
                    os.remove(file)
            print(" All iteration 4 results deleted")
        else:
            print(" Cancelled")
    else:
        print(" Results directory doesn't exist")

def full_reset():
    """Complete reset - checkpoint, embeddings, and results"""
    print("\n" + "="*60)
    print(" FULL RESET WARNING - ITERATION 4")
    print("="*60)
    response = input("This will delete EVERYTHING (checkpoint, embeddings, results). Continue? (yes/no): ")
    if response.lower() == 'yes':
        reset_checkpoint()
        if os.path.exists(EMBEDDINGS_BASE_DIR):
            shutil.rmtree(EMBEDDINGS_BASE_DIR)
            os.makedirs(EMBEDDINGS_BASE_DIR)
            print(" Embeddings deleted")
        if os.path.exists(RESULTS_DIR):
            for file in glob.glob(os.path.join(RESULTS_DIR, '*')):
                if os.path.isfile(file):
                    os.remove(file)
            print(" Results deleted")
        print("\n Full reset complete - Ready for fresh start")
    else:
        print(" Cancelled")

def remove_specific_model(model_key):
    """Remove specific model from checkpoint to reprocess it"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
            processed = data['processed']
        original_count = len(processed)
        processed = [item for item in processed if item[0] != model_key]
        removed_count = original_count - len(processed)
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({'processed': processed}, f)
        print(f" Removed {removed_count} combinations for model: {model_key}")
        print(f"Remaining: {len(processed)} combinations")
    else:
        print(" No checkpoint found")

def remove_specific_dataset(dataset_name):
    """Remove specific dataset from checkpoint to reprocess it"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
            processed = data['processed']
        original_count = len(processed)
        processed = [item for item in processed if item[1] != dataset_name]
        removed_count = original_count - len(processed)
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({'processed': processed}, f)
        print(f" Removed {removed_count} combinations for dataset: {dataset_name}")
        print(f"Remaining: {len(processed)} combinations")
    else:
        print(" No checkpoint found")

def complete_rerun():
    """Simple one-command complete restart for iteration 4"""
    print("\n" + "="*60)
    print(" COMPLETE RERUN - ITERATION 4")
    print("="*60)
    print("This will:")
    print("  1. Clear checkpoint file")
    print("  2. Delete all embeddings")
    print("  3. Delete all results")
    print("  4. Start fresh from beginning")
    print("="*60)
    response = input("\nProceed with complete rerun? (yes/no): ")
    if response.lower() == 'yes':
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
            print(" Checkpoint cleared")
        if os.path.exists(EMBEDDINGS_BASE_DIR):
            shutil.rmtree(EMBEDDINGS_BASE_DIR)
            os.makedirs(EMBEDDINGS_BASE_DIR)
            print(" Embeddings cleared")
        if os.path.exists(RESULTS_DIR):
            for file in glob.glob(os.path.join(RESULTS_DIR, '*')):
                if os.path.isfile(file):
                    os.remove(file)
            print(" Results cleared")
        print("\n Ready for complete rerun!")
        print("Run the main processing cell to start fresh.")
    else:
        print(" Cancelled")

# =============================================================================
# QUICK START GUIDE
# =============================================================================
print("""
============================================================
ITERATION 4 - RESET UTILITIES
============================================================
Available functions:
  view_progress()           - See what's been processed
  reset_checkpoint()        - Clear checkpoint to restart
  reset_embeddings()        - Delete all embeddings
  reset_results()           - Delete all results
  full_reset()              - Complete reset (all of above)
  complete_rerun()          - One-command complete restart
  remove_specific_model('model_name')   - Remove specific model
  remove_specific_dataset('dataset')    - Remove specific dataset

Run the next cell to start processing.
============================================================
""")

Found project root: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
BASE_DIR:            /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
DATA_DIR:            /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k
EMBEDDINGS_BASE_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Embeddings/_iteration_4
RESULTS_DIR:         /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_4
CHECKPOINT_FILE:     /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_4/checkpoint_iteration_4.json

ITERATION 4 - RESET UTILITIES
Available functions:
  view_progress()           - See what's been processed
  reset_checkpoint()        - Clear checkpoint to restart
  reset_embeddings()        - Delete all embeddings
  reset_results()           - Delete all results
  full_reset()              - Complete reset (all of above)
  complete_rerun()          - One-command complete restart
  remove_specific_mode

# 1. (Optional) Complete Rerun Command

Uncomment and execute the cell below **only** if a full reset of the Iteration 4 pipeline is required. This operation deletes all cached embeddings, results, and checkpoint files, forcing the pipeline to regenerate everything from scratch. Under normal circumstances, this cell should remain commented out to preserve existing work.

In [2]:
#complete_rerun()

# 2. Multi-Model Pipeline with SMOTE, Ensemble Methods, and Hyperparameter Tuning

This cell implements the complete Iteration 4 pipeline, encompassing library imports, enhanced configuration, and all core functions:

- **Text preprocessing**: Lowercasing, URL/email removal, and whitespace normalisation
- **Embedding generation**: Mean pooling with attention mask for contextualised sentence representations
- **SMOTE oversampling**: Synthetic minority oversampling to mitigate class imbalance
- **Classifier training**: LinearSVC (with GridSearchCV), Random Forest, XGBoost, LightGBM, and a Voting Ensemble of the top-3 classifiers
- **Comprehensive evaluation**: Per-classifier confusion matrices, ROC curves, precision-recall curves, and performance bar charts — each exported as a separate high-resolution figure (PNG + PDF, 300 dpi)

Embedding caching is enabled: if a `.pkl` file already exists for a given model-dataset combination, the pipeline loads it directly and skips redundant generation. Similarly, if all embeddings for a model are already cached, the transformer model is not loaded into memory.

In [3]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, precision_recall_fscore_support,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
import pickle
import glob
import gc
import json
import time
import warnings
warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost not available. Install with: pip install xgboost")
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    print("LightGBM not available. Install with: pip install lightgbm")
    LIGHTGBM_AVAILABLE = False

try:
    from imblearn.over_sampling import SMOTE
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("imbalanced-learn not available. Install with: pip install imbalanced-learn")
    IMBLEARN_AVAILABLE = False

# ============================================================
# ITERATION 4 - ENHANCED CONFIGURATION
# ============================================================

print("\n" + "#"*80)
print("ITERATION 4: ENHANCED MULTI-MODEL PIPELINE")
print("Advanced Techniques: SMOTE, Mean Pooling, Ensemble Methods, Hyperparameter Tuning")
print("#"*80)

# All paths are defined in Cell 2 via find_project_root_with_datasets().
# DATA_DIR, EMBEDDINGS_BASE_DIR, RESULTS_DIR, CHECKPOINT_FILE, LANGUAGE_CODE_MAP
# are set there — no hardcoded paths needed here.
print(f"\nUsing paths from Cell 2 (standardized):")
print(f"   DATA_DIR:            {DATA_DIR}")
print(f"   EMBEDDINGS_BASE_DIR: {EMBEDDINGS_BASE_DIR}")
print(f"   RESULTS_DIR:         {RESULTS_DIR}")
print(f"   CHECKPOINT_FILE:     {CHECKPOINT_FILE}")

# Create directories
os.makedirs(EMBEDDINGS_BASE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CHECKPOINT_FILE), exist_ok=True)

# Enhanced configuration
CONFIG = {
    'pooling_strategy': 'mean',  # 'cls', 'mean', or 'max'
    'use_smote': True,
    'smote_strategy': 0.7,  # Oversample minority to 70% of majority
    'use_feature_scaling': True,
    'hyperparameter_tuning': True,   # SVM GridSearchCV only
    'use_ensemble': True,
    'classifiers': ['svm', 'xgboost', 'lightgbm', 'random_forest', 'ensemble'],
    'cv_folds': 5,
    'text_preprocessing': True,
}

_RQ1_BASELINE = 0.7825  # Iteration 0: bert-base-uncased + SVM → Macro F1

# Model configurations - Best performing models with FIXED paths
MODELS = {
    'bert-base-multilingual-uncased': 'google-bert/bert-base-multilingual-uncased',
    'xlm-roberta-base': 'xlm-roberta-base', 
    'paraphrase-multilingual-mpnet-base-v2': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'paraphrase-multilingual-MiniLM-L12-v2': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'snowflake-arctic-embed-l-v2.0': 'snowflake/snowflake-arctic-embed-l-v2.0',
    'upskyy/bge-m3-korean': 'upskyy/bge-m3-korean',
    'Alibaba-NLP/gte-multilingual-base': 'Alibaba-NLP/gte-multilingual-base',
    'intfloat/multilingual-e5-large-instruct': 'intfloat/multilingual-e5-large-instruct',
}


# Datasets to skip
SKIP_DATASETS = [
    'master_df_Hungary_manual.json',
    'master_df_International_manual.json',
    'master_df_Russia_manual.json',
    'master_df_UAE_manual.json',
    'master_df_Poland_manual.json',
    'master_df_Unknown_manual.json',
]

# Processing settings
BATCH_SIZE = 12
MAX_SAMPLES_SVM = 6000

print(f"\n Configuration:")
print(f"   Pooling Strategy: {CONFIG['pooling_strategy']}")
print(f"   Use SMOTE: {CONFIG['use_smote']}")
print(f"   Hyperparameter Tuning: {CONFIG['hyperparameter_tuning']}")
print(f"   Ensemble Methods: {CONFIG['use_ensemble']}")
print(f"   Models: {list(MODELS.keys())}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Max SVM Training Samples: {MAX_SAMPLES_SVM}")

# ============================================================
# TEXT PREPROCESSING FUNCTIONS
# ============================================================

def advanced_text_preprocessing(text):
    """Enhanced text preprocessing"""
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters but keep sentence structure
    text = re.sub(r'[^a-zA-Z0-9\s\.]', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================

def save_checkpoint(processed_items):
    """Save checkpoint with processed model-dataset combinations"""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'processed': list(processed_items)}, f)
    print(f"Checkpoint saved: {len(processed_items)} combinations processed")

def load_checkpoint():
    """Load checkpoint"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return set(tuple(item) for item in json.load(f)['processed'])
    return set()

def reset_checkpoint():
    """Delete checkpoint to restart"""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint cleared")
    else:
        print("No checkpoint found")

# ============================================================
# ENHANCED EMBEDDING GENERATION
# ============================================================

def get_enhanced_embeddings(texts, tokenizer, model, pooling='mean', max_length=512, batch_size=12):
    """
    Generate embeddings with advanced pooling strategies
    
    Args:
        pooling: 'cls' (first token), 'mean' (mean pooling), or 'max' (max pooling)
    """
    embeddings = []
    device = 'mps' if torch.backends.mps.is_available() else 'cpu'
    model = model.to(device)
    model.eval()
    
    print(f"Using device: {device}")
    print(f"Pooling strategy: {pooling}")
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i + batch_size]
        
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            
            if pooling == 'cls':
                # [CLS] token (first token)
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                
            elif pooling == 'mean':
                # Mean pooling with attention mask
                attention_mask = inputs['attention_mask']
                token_embeddings = outputs.last_hidden_state
                input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
                sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
                sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
                batch_embeddings = (sum_embeddings / sum_mask).cpu().numpy()
                
            elif pooling == 'max':
                # Max pooling
                token_embeddings = outputs.last_hidden_state
                batch_embeddings = torch.max(token_embeddings, dim=1)[0].cpu().numpy()
                
            else:
                raise ValueError(f"Unknown pooling strategy: {pooling}")
            
            embeddings.extend(batch_embeddings)
        
        # Clear GPU memory after each batch
        if device == 'cuda':
            torch.cuda.empty_cache()
        elif device == 'mps' and hasattr(torch.mps, 'empty_cache'):
            torch.mps.empty_cache()
        
        # Clear CPU memory periodically
        if i % 50 == 0 and i > 0:
            gc.collect()
    
    return np.array(embeddings)

# ============================================================
# DATASET PROCESSING WITH ENHANCEMENTS
# ============================================================

def find_cached_pkl(model_key, dataset_name):
    """Find cached embedding file, checking both current and legacy naming conventions."""
    model_dir = os.path.join(EMBEDDINGS_BASE_DIR, model_key.replace('/', '_'))
    # Try current naming
    pkl_file = os.path.join(model_dir, f'{dataset_name}_{model_key.replace("/", "_")}_embeddings.pkl')
    if os.path.exists(pkl_file):
        return pkl_file
    # Try legacy naming (iteration 4 originally used dutch/german/swedish)
    alt_names = {
        'germany_manual': 'german_manual',
        'sweden_manual': 'swedish_manual',
        'netherlands_manual': 'dutch_manual',
    }
    alt_name = alt_names.get(dataset_name)
    if alt_name:
        alt_pkl = os.path.join(model_dir, f'{alt_name}_{model_key.replace("/", "_")}_embeddings.pkl')
        if os.path.exists(alt_pkl):
            return alt_pkl
    return None

def process_dataset_enhanced(json_file, model_key, model_name, tokenizer, model, dataset_name=None):
    """Process dataset with enhanced preprocessing and embeddings"""
    
    # dataset_name comes from the json_files dict key (e.g. 'english_manual')
    if dataset_name is None:
        dataset_name = os.path.basename(json_file).replace('master_df_', '').replace('.json', '').lower()
    
    # Check if embeddings are already cached
    cached_pkl = find_cached_pkl(model_key, dataset_name)
    if cached_pkl:
        print(f"\n[SKIP] Embeddings already cached for {dataset_name} + {model_key}")
        print(f"       {cached_pkl}")
        return cached_pkl
    
    print(f"\n{'='*80}")
    print(f"Processing: {dataset_name.upper()}")
    print(f"Model: {model_key}")
    print(f"{'='*80}")
    
    # Load dataset from Iteration 0 pre-processed JSON (no re-preprocessing needed)
    df = pd.read_json(json_file)
    if 'LANGUAGE' in df.columns:
        df['LANGUAGE_CODE'] = df['LANGUAGE'].map(LANGUAGE_CODE_MAP).fillna('UN')
    print(f"Dataset shape: {df.shape}")
    
    # Create binary labels
    df['binary_label'] = df['CASE_TYPE'].apply(lambda x: 1 if x == 'Process Safety' else 0)
    
    print(f"\nCase type distribution:")
    print(df['CASE_TYPE'].value_counts())
    print(f"\nClass balance:")
    print(df['binary_label'].value_counts(normalize=True))
    
    # Enhanced text preprocessing
    if CONFIG['text_preprocessing']:
        print(f"\nApplying advanced text preprocessing...")
        df['text_features'] = df.apply(
            lambda x: advanced_text_preprocessing(str(x['TITLE']) + '. ' + str(x['CASE_DESCRIPTION'])),
            axis=1
        )
    else:
        df['text_features'] = df['TITLE'] + '. ' + df['CASE_DESCRIPTION']
    
    # Generate embeddings with selected pooling strategy
    print(f"\nGenerating {model_name} embeddings...")
    X = get_enhanced_embeddings(
        df['text_features'].tolist(),
        tokenizer,
        model,
        pooling=CONFIG['pooling_strategy'],
        batch_size=BATCH_SIZE
    )
    y = df['binary_label'].values
    
    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"\nTraining set: {X_train.shape}")
    print(f"Test set: {X_test.shape}")
    
    # Apply SMOTE if enabled
    if CONFIG['use_smote'] and IMBLEARN_AVAILABLE:
        print(f"\n Applying SMOTE for class balancing...")
        print(f"   Strategy: Oversample minority to {CONFIG['smote_strategy']*100}% of majority")
        
        try:
            smote = SMOTE(
                sampling_strategy=CONFIG['smote_strategy'],
                random_state=42,
                k_neighbors=min(5, sum(y_train == 1) - 1)
            )
            X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
            
            print(f"   Original training size: {X_train.shape[0]}")
            print(f"   Balanced training size: {X_train_balanced.shape[0]}")
            print(f"   Original class distribution: {np.bincount(y_train)}")
            print(f"   Balanced class distribution: {np.bincount(y_train_balanced)}")
            
            X_train_final = X_train_balanced
            y_train_final = y_train_balanced
            used_smote = True
            
        except Exception as e:
            print(f"   SMOTE failed: {e}")
            print(f"   Using original training data")
            X_train_final = X_train
            y_train_final = y_train
            used_smote = False
    else:
        X_train_final = X_train
        y_train_final = y_train
        used_smote = False
    
    # Subsample if still too large for SVM
    if X_train_final.shape[0] > MAX_SAMPLES_SVM:
        print(f"\n Training set too large ({X_train_final.shape[0]} samples)")
        print(f"   Subsampling to {MAX_SAMPLES_SVM} for SVM efficiency...")
        from sklearn.utils import resample
        X_train_subset, y_train_subset = resample(
            X_train_final, y_train_final,
            n_samples=MAX_SAMPLES_SVM,
            stratify=y_train_final,
            random_state=42
        )
        print(f"   Subsampled training set: {X_train_subset.shape}")
        subsampled = True
    else:
        X_train_subset = X_train_final
        y_train_subset = y_train_final
        subsampled = False
    
    # Feature scaling — fit on original (pre-SMOTE) X_train to avoid data leakage
    if CONFIG['use_feature_scaling']:
        print(f"\n Applying feature scaling...")
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train_subset_scaled = scaler.transform(X_train_subset)
        X_test_scaled = scaler.transform(X_test)
    else:
        X_train_subset_scaled = X_train_subset
        X_test_scaled = X_test
        scaler = None
    
    # Create model-specific directory
    model_dir = os.path.join(EMBEDDINGS_BASE_DIR, model_key.replace('/', '_'))
    os.makedirs(model_dir, exist_ok=True)
    
    # Save embeddings and preprocessing artifacts
    embeddings_data = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'X_train_balanced': X_train_final,
        'y_train_balanced': y_train_final,
        'X_train_subset': X_train_subset_scaled,
        'y_train_subset': y_train_subset,
        'X_test_scaled': X_test_scaled,
        'scaler': scaler,
        'feature_names': df['text_features'].tolist(),
        'metadata': {
            'model': model_name,
            'model_key': model_key,
            'dataset': dataset_name,
            'embedding_dim': X_train.shape[1],
            'train_size_original': X_train.shape[0],
            'train_size_balanced': X_train_final.shape[0],
            'train_size_final': X_train_subset.shape[0],
            'test_size': X_test.shape[0],
            'random_state': 42,
            'pooling_strategy': CONFIG['pooling_strategy'],
            'used_smote': used_smote,
            'smote_strategy': CONFIG['smote_strategy'] if used_smote else None,
            'subsampled': subsampled,
            'feature_scaling': CONFIG['use_feature_scaling'],
            'text_preprocessing': CONFIG['text_preprocessing'],
        }
    }
    
    pkl_file = os.path.join(model_dir, f'{dataset_name}_{model_key.replace("/", "_")}_embeddings.pkl')
    
    with open(pkl_file, 'wb') as f:
        pickle.dump(embeddings_data, f)
    
    print(f"\n{'='*60}")
    print(f"Embeddings saved!")
    print(f"Location: {pkl_file}")
    print(f"Size: {os.path.getsize(pkl_file) / 1024**2:.2f} MB")
    print(f"{'='*60}\n")
    
    # Clear memory
    del df, X, y, X_train, X_test, y_train, y_test
    if used_smote:
        del X_train_balanced, y_train_balanced
    del X_train_final, y_train_final, X_train_subset, y_train_subset
    del X_train_subset_scaled, X_test_scaled, embeddings_data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch, 'mps') and hasattr(torch.mps, 'empty_cache'):
        torch.mps.empty_cache()
    
    return pkl_file

# ============================================================
# ENHANCED CLASSIFIER TRAINING
# ============================================================

def train_multiple_classifiers(X_train, y_train, X_test, y_test, dataset_name, model_key):
    """Train and evaluate multiple classifiers"""
    
    results = {}
    
    print(f"\n{'='*80}")
    print(f"Training Multiple Classifiers: {dataset_name.upper()}")
    print(f"{'='*80}")
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Test samples: {X_test.shape[0]}")
    print(f"Features: {X_train.shape[1]}")
    
    # Define classifiers
    classifiers = {}
    
    # SVM with hyperparameter tuning
    if 'svm' in CONFIG['classifiers']:
        if CONFIG['hyperparameter_tuning']:
            print(f"\n Training SVM with hyperparameter tuning...")
            param_grid_svm = {
                'C': [0.1, 1, 10, 100],
                'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}],
                'C': [1, 10],
                'class_weight': ['balanced', {0: 1, 1: 3}],
            }
            svm_base = LinearSVC(dual=False, random_state=42, max_iter=10000)
            svm_model = GridSearchCV(
                svm_base, param_grid_svm, cv=3, scoring='f1_macro',
                n_jobs=-1, verbose=0
            )
        else:
            svm_model = LinearSVC(class_weight='balanced', max_iter=10000, dual=False, random_state=42)
        
        classifiers['SVM'] = svm_model
    
    # Random Forest
    if 'random_forest' in CONFIG['classifiers']:
        print(f"\n Training Random Forest...")
        classifiers['Random_Forest'] = RandomForestClassifier(
            n_estimators=100,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
            max_depth=20
        )
    
    # XGBoost
    if 'xgboost' in CONFIG['classifiers'] and XGBOOST_AVAILABLE:
        print(f"\n Training XGBoost...")
        pos_weight = sum(y_train == 0) / sum(y_train == 1)
        classifiers['XGBoost'] = XGBClassifier(
            n_estimators=100,
            scale_pos_weight=pos_weight,
            random_state=42,
            n_jobs=-1,
            max_depth=6,
            learning_rate=0.1
        )
    
    # LightGBM
    if 'lightgbm' in CONFIG['classifiers'] and LIGHTGBM_AVAILABLE:
        print(f"\n Training LightGBM...")
        classifiers['LightGBM'] = LGBMClassifier(
            n_estimators=100,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
            max_depth=6,
            learning_rate=0.1,
            verbose=-1
        )
    
    # Train each classifier
    for name, clf in classifiers.items():
        print(f"\n{'─'*60}")
        print(f" {name}")
        print(f"{'─'*60}")
        
        start_time = time.time()
        clf.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        y_pred = clf.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        f1_weighted = f1_score(y_test, y_pred, average='weighted')
        f1_macro = f1_score(y_test, y_pred, average='macro')
        
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, labels=[0, 1]
        )
        
        # Try to get probability predictions for AUC
        try:
            if hasattr(clf, 'predict_proba'):
                y_proba = clf.predict_proba(X_test)[:, 1]
            elif hasattr(clf, 'decision_function'):
                y_proba = clf.decision_function(X_test)
            else:
                y_proba = None
                
            if y_proba is not None:
                roc_auc = roc_auc_score(y_test, y_proba)
                avg_precision = average_precision_score(y_test, y_proba)
            else:
                roc_auc = None
                avg_precision = None
        except:
            y_proba = None
            roc_auc = None
            avg_precision = None
        
        # Store results
        results[name] = {
            'model': clf,
            'y_pred': y_pred,
            'y_proba': y_proba,
            'accuracy': accuracy,
            'f1_weighted': f1_weighted,
            'f1_macro': f1_macro,
            'precision_non_ps': precision[0],
            'recall_non_ps': recall[0],
            'f1_non_ps': f1[0],
            'support_non_ps': support[0],
            'precision_ps': precision[1],
            'recall_ps': recall[1],
            'f1_ps': f1[1],
            'support_ps': support[1],
            'training_time': training_time,
            'roc_auc': roc_auc,
            'avg_precision': avg_precision,
        }
        
        # Print best parameters if GridSearchCV was used
        if hasattr(clf, 'best_params_'):
            print(f"   Best parameters: {clf.best_params_}")
            print(f"   Best CV score: {clf.best_score_:.4f}")
        
        print(f"   Accuracy: {accuracy:.4f}")
        print(f"   F1 (weighted): {f1_weighted:.4f}")
        print(f"   F1 (Process Safety): {f1[1]:.4f}")
        if roc_auc:
            print(f"   ROC AUC: {roc_auc:.4f}")
        print(f"   Training time: {training_time:.2f}s")
    
    # Create ensemble if enabled
    if CONFIG['use_ensemble'] and len(classifiers) >= 2:
        print(f"\n{'─'*60}")
        print(f" Creating Voting Ensemble")
        print(f"{'─'*60}")
        
        try:
            # Select top 3 classifiers by Process Safety F1
            sorted_by_f1ps = sorted(results.keys(), key=lambda x: results[x]['f1_ps'], reverse=True)
            top_3_names = sorted_by_f1ps[:3]
            ensemble_classifiers = [(name, classifiers[name]) for name in top_3_names if name in classifiers]
            
            voting_clf = VotingClassifier(
                estimators=ensemble_classifiers,
                voting='hard'
            )
            
            start_time = time.time()
            voting_clf.fit(X_train, y_train)
            training_time = time.time() - start_time
            
            y_pred = voting_clf.predict(X_test)
            
            accuracy = accuracy_score(y_test, y_pred)
            f1_weighted = f1_score(y_test, y_pred, average='weighted')
            f1_macro = f1_score(y_test, y_pred, average='macro')
            
            precision, recall, f1, support = precision_recall_fscore_support(
                y_test, y_pred, average=None, labels=[0, 1]
            )
            
            results['Voting_Ensemble'] = {
                'model': voting_clf,
                'y_pred': y_pred,
                'y_proba': None,
                'accuracy': accuracy,
                'f1_weighted': f1_weighted,
                'f1_macro': f1_macro,
                'precision_non_ps': precision[0],
                'recall_non_ps': recall[0],
                'f1_non_ps': f1[0],
                'support_non_ps': support[0],
                'precision_ps': precision[1],
                'recall_ps': recall[1],
                'f1_ps': f1[1],
                'support_ps': support[1],
                'training_time': training_time,
                'roc_auc': None,
                'avg_precision': None,
            }
            
            print(f"   Accuracy: {accuracy:.4f}")
            print(f"   F1 (weighted): {f1_weighted:.4f}")
            print(f"   F1 (Process Safety): {f1[1]:.4f}")
            print(f"   Training time: {training_time:.2f}s")
            
        except Exception as e:
            print(f"   Ensemble creation failed: {e}")
    
    # Find best classifier
    best_classifier_name = max(results, key=lambda x: results[x]['f1_ps'])
    print(f"\n{'='*80}")
    print(f"Best Classifier: {best_classifier_name}")
    print(f"   Process Safety F1: {results[best_classifier_name]['f1_ps']:.4f}")
    print(f"{'='*80}")
    
    return results, best_classifier_name

# ============================================================
# COMPREHENSIVE EVALUATION
# ============================================================

def comprehensive_evaluation(results, X_test, y_test, dataset_name, model_key):
    """Comprehensive evaluation with separate academic-quality figures"""
    
    all_results = []
    
    for clf_name, result in results.items():
        y_pred = result['y_pred']
        y_proba = result['y_proba']
        
        print(f"\n{'='*80}")
        print(f"Detailed Report: {dataset_name.upper()} | {model_key} | {clf_name}")
        print(f"{'='*80}")
        
        # Classification report
        print("\nClassification Report:")
        print(classification_report(
            y_test, y_pred,
            target_names=['Non-Process Safety', 'Process Safety'],
            digits=4
        ))
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        
        safe_model = model_key.replace('/', '_')
        base_name = f"{dataset_name}_{safe_model}_{clf_name}"
        
        # --- Academic rcParams ---
        _original_rc = {k: plt.rcParams.get(k) for k in [
            'font.family', 'font.serif', 'axes.titlesize', 'axes.labelsize',
            'xtick.labelsize', 'ytick.labelsize']}
        plt.rcParams.update({
            'font.family': 'serif',
            'font.serif': ['Times New Roman', 'DejaVu Serif'],
            'axes.titlesize': 18, 'axes.labelsize': 16,
            'xtick.labelsize': 13, 'ytick.labelsize': 13,
        })
        
        # ---- 1. CONFUSION MATRIX (separate figure) ----
        fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Process Safety', 'Process Safety'],
            yticklabels=['Non-Process Safety', 'Process Safety'],
            ax=ax_cm, linewidths=0.8, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Count'},
            annot_kws={'size': 18, 'fontweight': 'bold'},
        )
        ax_cm.set_title(
            f'Confusion Matrix — {dataset_name.upper()}\n({model_key}, {clf_name})',
            fontsize=18, fontweight='bold', pad=15)
        ax_cm.set_ylabel('True Label', fontsize=16)
        ax_cm.set_xlabel('Predicted Label', fontsize=16)
        fig_cm.tight_layout()
        cm_path = os.path.join(RESULTS_DIR, f'confusion_matrix_{base_name}')
        fig_cm.savefig(cm_path + '.png', dpi=300, bbox_inches='tight')
        fig_cm.savefig(cm_path + '.pdf', bbox_inches='tight')
        print(f"[OK] Saved confusion matrix PNG + PDF: {os.path.basename(cm_path)}")
        plt.show()
        plt.close(fig_cm)
        
        # ---- 2. ROC CURVE (separate figure, if probabilities available) ----
        if y_proba is not None:
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
            ax_roc.plot(fpr, tpr, linewidth=2.2,
                        label=f'ROC (AUC = {result["roc_auc"]:.3f})')
            ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
            ax_roc.set_xlabel('False Positive Rate', fontsize=16)
            ax_roc.set_ylabel('True Positive Rate', fontsize=16)
            ax_roc.set_title(
                f'ROC Curve — {dataset_name.upper()}\n({model_key}, {clf_name})',
                fontsize=18, fontweight='bold', pad=15)
            ax_roc.legend(fontsize=13, loc='lower right')
            ax_roc.grid(alpha=0.3)
            fig_roc.tight_layout()
            roc_path = os.path.join(RESULTS_DIR, f'roc_curve_{base_name}')
            fig_roc.savefig(roc_path + '.png', dpi=300, bbox_inches='tight')
            fig_roc.savefig(roc_path + '.pdf', bbox_inches='tight')
            print(f"[OK] Saved ROC curve PNG + PDF: {os.path.basename(roc_path)}")
            plt.show()
            plt.close(fig_roc)
            
            # ---- 3. PRECISION-RECALL CURVE (separate figure) ----
            precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)
            fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
            ax_pr.plot(recall_curve, precision_curve, linewidth=2.2,
                       label=f'AP = {result["avg_precision"]:.3f}')
            ax_pr.set_xlabel('Recall', fontsize=16)
            ax_pr.set_ylabel('Precision', fontsize=16)
            ax_pr.set_title(
                f'Precision-Recall Curve — {dataset_name.upper()}\n({model_key}, {clf_name})',
                fontsize=18, fontweight='bold', pad=15)
            ax_pr.legend(fontsize=13, loc='upper right')
            ax_pr.grid(alpha=0.3)
            fig_pr.tight_layout()
            pr_path = os.path.join(RESULTS_DIR, f'precision_recall_{base_name}')
            fig_pr.savefig(pr_path + '.png', dpi=300, bbox_inches='tight')
            fig_pr.savefig(pr_path + '.pdf', bbox_inches='tight')
            print(f"[OK] Saved PR curve PNG + PDF: {os.path.basename(pr_path)}")
            plt.show()
            plt.close(fig_pr)
        
        # ---- 4. METRICS BAR CHART (separate figure) ----
        metrics_data = {
            'Accuracy': result['accuracy'],
            'F1 (weighted)': result['f1_weighted'],
            'F1 (PS)': result['f1_ps'],
            'Precision (PS)': result['precision_ps'],
            'Recall (PS)': result['recall_ps'],
        }
        fig_bar, ax_bar = plt.subplots(figsize=(8, 6))
        bars = ax_bar.barh(list(metrics_data.keys()), list(metrics_data.values()),
                           color='steelblue', alpha=0.85, edgecolor='white')
        ax_bar.set_xlim([0, 1.05])
        ax_bar.set_title(
            f'Performance Metrics — {dataset_name.upper()}\n({model_key}, {clf_name})',
            fontsize=18, fontweight='bold', pad=15)
        ax_bar.set_xlabel('Score', fontsize=16)
        ax_bar.grid(axis='x', alpha=0.3)
        for i, (k, v) in enumerate(metrics_data.items()):
            ax_bar.text(v + 0.015, i, f'{v:.4f}', va='center', fontsize=12)
        fig_bar.tight_layout()
        bar_path = os.path.join(RESULTS_DIR, f'metrics_bar_{base_name}')
        fig_bar.savefig(bar_path + '.png', dpi=300, bbox_inches='tight')
        fig_bar.savefig(bar_path + '.pdf', bbox_inches='tight')
        print(f"[OK] Saved metrics bar PNG + PDF: {os.path.basename(bar_path)}")
        plt.show()
        plt.close(fig_bar)
        
        # Restore rcParams
        plt.rcParams.update({k: v for k, v in _original_rc.items() if v is not None})
        
        # Compile results
        result_dict = {
            'dataset': dataset_name,
            'model': model_key,
            'classifier': clf_name,
            'accuracy': result['accuracy'],
            'f1_weighted': result['f1_weighted'],
            'f1_macro': result['f1_macro'],
            'precision_non_ps': result['precision_non_ps'],
            'recall_non_ps': result['recall_non_ps'],
            'f1_non_ps': result['f1_non_ps'],
            'support_non_ps': result['support_non_ps'],
            'precision_ps': result['precision_ps'],
            'recall_ps': result['recall_ps'],
            'f1_ps': result['f1_ps'],
            'support_ps': result['support_ps'],
            'training_time': result['training_time'],
            'roc_auc': result['roc_auc'],
            'avg_precision': result['avg_precision'],
            'confusion_matrix': cm.tolist(),
        }
        
        all_results.append(result_dict)
    
    return all_results

# ============================================================
# MAIN TRAINING PIPELINE
# ============================================================

def train_and_evaluate_embeddings(embeddings_file):
    """Load embeddings and train multiple classifiers"""
    
    # Load embeddings
    with open(embeddings_file, 'rb') as f:
        embeddings_data = pickle.load(f)
    
    X_train = embeddings_data['X_train_subset']
    y_train = embeddings_data['y_train_subset']
    X_test = embeddings_data['X_test_scaled']
    y_test = embeddings_data['y_test']
    metadata = embeddings_data['metadata']
    
    dataset_name = metadata['dataset']
    model_key = metadata['model_key']
    
    print(f"\n{'#'*80}")
    print(f" Training Pipeline: {dataset_name.upper()}")
    print(f"{'#'*80}")
    print(f"Model: {model_key}")
    print(f"Pooling: {metadata['pooling_strategy']}")
    print(f"SMOTE used: {metadata['used_smote']}")
    print(f"Feature scaling: {metadata['feature_scaling']}")
    
    # Train multiple classifiers
    results, best_classifier = train_multiple_classifiers(
        X_train, y_train, X_test, y_test, dataset_name, model_key
    )
    
    # Comprehensive evaluation
    all_results = comprehensive_evaluation(
        results, X_test, y_test, dataset_name, model_key
    )
    
    # Clear memory
    del embeddings_data, X_train, X_test, y_train, y_test
    gc.collect()
    
    return all_results

# ============================================================
# MAIN EXECUTION
# ============================================================

print("\n" + "#"*80)
print(" STARTING ITERATION 4 PIPELINE")
print("#"*80)

# =============================================================================
# Load datasets from pre-processed master dataset splits
# =============================================================================

DATASET_FILES = {
    'english_manual':     str(PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_English_manual.json'),
    'germany_manual':     str(PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Germany_manual.json'),
    'sweden_manual':      str(PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Sweden_manual.json'),
    'netherlands_manual': str(PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Netherlands_manual.json'),
}

# Only include files that actually exist
json_files = {name: path for name, path in DATASET_FILES.items() if os.path.exists(path)}
missing = [name for name, path in DATASET_FILES.items() if not os.path.exists(path)]

print(f"\n Found {len(json_files)} datasets to process")
if missing:
    print(f" Skipping missing: {missing}")
print(f" Total combinations: {len(json_files) * len(MODELS)}")

# Load checkpoint
processed_items = load_checkpoint()
print(f"\n Already processed: {len(processed_items)} combinations")

# Track results
all_results = []
start_time = time.time()

# Process each model
for model_key, model_name in MODELS.items():
    print(f"\n{'#'*80}")
    print(f" MODEL: {model_key}")
    print(f"{'#'*80}")
    
    try:
        # Check if all embeddings are cached for this model
        all_cached = True
        cached_pkls = []
        for ds_name in json_files:
            cached = find_cached_pkl(model_key, ds_name)
            if cached:
                cached_pkls.append(cached)
            else:
                all_cached = False
        
        if all_cached:
            print(f"\n[SKIP] All {len(json_files)} embeddings cached for {model_key} — skipping model load.")
            pkl_files = cached_pkls
        else:
            # Initialize model (only when some embeddings are missing)
            print(f"\n Initializing {model_key}...")
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
            
            # Add padding token if missing
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
                if hasattr(model.config, 'eos_token_id'):
                    model.config.pad_token_id = model.config.eos_token_id
            
            print(f" Model loaded successfully!")
        
        # Phase 1: Generate embeddings (only if not all cached)
        if not all_cached:
            print(f"\n{'='*80}")
            print(f" PHASE 1: GENERATING EMBEDDINGS - {model_key}")
            print(f"{'='*80}")
            
            pkl_files = []
            
            for dataset_name, json_file in json_files.items():
                # dataset_name is already the dict key (e.g. 'english_manual')
                item_key = (model_key, dataset_name)
                
                # Check if already processed
                if item_key in processed_items:
                    print(f"\n Skipping already processed: {dataset_name} with {model_key}")
                    cached = find_cached_pkl(model_key, dataset_name)
                    if cached:
                        pkl_files.append(cached)
                    continue
                
                try:
                    pkl_file = process_dataset_enhanced(json_file, model_key, model_name, tokenizer, model, dataset_name=dataset_name)
                    pkl_files.append(pkl_file)
                    
                    # Save checkpoint
                    processed_items.add(item_key)
                    save_checkpoint(processed_items)
                    
                    print(f" Progress: {len(processed_items)}/{len(json_files) * len(MODELS)}")
                   
                except Exception as e:
                    print(f" ERROR processing {dataset_name} with {model_key}: {str(e)}\n")
                    import traceback
                    traceback.print_exc()
                    continue
        
        # Phase 2: Train classifiers
        print(f"\n{'='*80}")
        print(f" PHASE 2: TRAINING CLASSIFIERS - {model_key}")
        print(f"{'='*80}")
        
        for pkl_file in pkl_files:
            try:
                results = train_and_evaluate_embeddings(pkl_file)
                all_results.extend(results)
                
            except Exception as e:
                dataset_name = os.path.basename(pkl_file).split('_')[0]
                print(f" ERROR training classifiers for {dataset_name} with {model_key}: {str(e)}\n")
                import traceback
                traceback.print_exc()
                continue
        
        # Clear model from memory (only if it was loaded)
        if not all_cached:
            del tokenizer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif hasattr(torch, 'mps') and hasattr(torch.mps, 'empty_cache'):
            torch.mps.empty_cache()
        
        print(f"\n Completed processing {model_key}")
        
    except Exception as e:
        print(f"\n ERROR loading model {model_key}: {str(e)}")
        import traceback
        traceback.print_exc()
        print(f" Skipping this model and continuing...\n")
        continue

# ============================================================
# FINAL COMPREHENSIVE SUMMARY
# ============================================================

print("\n" + "#"*80)
print(" FINAL COMPREHENSIVE SUMMARY - ITERATION 4")
print("#"*80 + "\n")

if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Display results table
    print("="*100)
    print(" DETAILED RESULTS TABLE")
    print("="*100)
    
    detailed_cols = [
        'dataset', 'model', 'classifier', 'accuracy', 'f1_weighted', 'f1_macro',
        'precision_ps', 'recall_ps', 'f1_ps', 'support_ps',
        'roc_auc', 'avg_precision', 'training_time'
    ]
    
    display_cols = [col for col in detailed_cols if col in results_df.columns]
    display_df = results_df[display_cols].copy()
    
    # Round numeric columns
    numeric_cols = ['accuracy', 'f1_weighted', 'f1_macro', 
                    'precision_ps', 'recall_ps', 'f1_ps',
                    'roc_auc', 'avg_precision']
    
    for col in numeric_cols:
        if col in display_df.columns:
            display_df[col] = display_df[col].round(4)
    
    if 'training_time' in display_df.columns:
        display_df['training_time'] = display_df['training_time'].round(2)
    
    print(display_df.to_string(index=False))
    
    # Save results to JSON
    summary_file = os.path.join(RESULTS_DIR, 'iteration_4_comprehensive_results.json')
    results_df.to_json(summary_file, orient='records', lines=True)
    print(f"\n Results saved to: {summary_file}")
    
    # Aggregate statistics
    print(f"\n{'='*80}")
    print(" AGGREGATE STATISTICS")
    print(f"{'='*80}")
    print(f"Total model-dataset-classifier combinations: {len(results_df)}")
    print(f"Datasets processed: {results_df['dataset'].nunique()}")
    print(f"Models used: {results_df['model'].nunique()}")
    print(f"Classifiers tested: {results_df['classifier'].nunique()}")
    
    if results_df['accuracy'].notna().any():
        print(f"\n Overall Performance Statistics:")
        print(f"{'─'*60}")
        
        metrics_summary = {
            'Accuracy': results_df['accuracy'],
            'F1 (weighted)': results_df['f1_weighted'],
            'F1 (macro)': results_df['f1_macro'],
            'F1 (Process Safety)': results_df['f1_ps'],
            'Precision (PS)': results_df['precision_ps'],
            'Recall (PS)': results_df['recall_ps'],
        }
        
        for metric_name, metric_values in metrics_summary.items():
            print(f"\n{metric_name}:")
            print(f"  Min:  {metric_values.min():.4f}")
            print(f"  Max:  {metric_values.max():.4f}")
            print(f"  Mean: {metric_values.mean():.4f}")
            print(f"  Std:  {metric_values.std():.4f}")
        
        if 'roc_auc' in results_df.columns:
            roc_auc_valid = results_df['roc_auc'].dropna()
            if len(roc_auc_valid) > 0:
                print(f"\nROC AUC:")
                print(f"  Min:  {roc_auc_valid.min():.4f}")
                print(f"  Max:  {roc_auc_valid.max():.4f}")
                print(f"  Mean: {roc_auc_valid.mean():.4f}")
        
        if 'training_time' in results_df.columns:
            print(f"\n Training Time Statistics:")
            print(f"  Min:   {results_df['training_time'].min():.2f}s")
            print(f"  Max:   {results_df['training_time'].max():.2f}s")
            print(f"  Mean:  {results_df['training_time'].mean():.2f}s")
            print(f"  Total: {results_df['training_time'].sum():.2f}s")
        
        # Best performing combination
        best_idx = results_df['f1_ps'].idxmax()
        print(f"\n{'='*80}")
        print(" BEST PERFORMING COMBINATION (by Process Safety F1)")
        print(f"{'='*80}")
        print(f"Dataset:    {results_df.loc[best_idx, 'dataset']}")
        print(f"Model:      {results_df.loc[best_idx, 'model']}")
        print(f"Classifier: {results_df.loc[best_idx, 'classifier']}")
        print(f"\n Metrics:")
        print(f"  Accuracy:           {results_df.loc[best_idx, 'accuracy']:.4f}")
        print(f"  F1 (weighted):      {results_df.loc[best_idx, 'f1_weighted']:.4f}")
        print(f"  F1 (PS):            {results_df.loc[best_idx, 'f1_ps']:.4f}")
        print(f"  Precision (PS):     {results_df.loc[best_idx, 'precision_ps']:.4f}")
        print(f"  Recall (PS):        {results_df.loc[best_idx, 'recall_ps']:.4f}")
        if pd.notna(results_df.loc[best_idx, 'roc_auc']):
            print(f"  ROC AUC:            {results_df.loc[best_idx, 'roc_auc']:.4f}")
        
        # Average performance by model
        print(f"\n{'='*80}")
        print(" AVERAGE PERFORMANCE BY MODEL")
        print(f"{'='*80}")
        model_stats = results_df.groupby('model')[
            ['accuracy', 'f1_weighted', 'f1_ps', 'precision_ps', 'recall_ps']
        ].agg(['mean', 'std'])
        print(model_stats.to_string())
        
        # Average performance by classifier
        print(f"\n{'='*80}")
        print(" AVERAGE PERFORMANCE BY CLASSIFIER")
        print(f"{'='*80}")
        classifier_stats = results_df.groupby('classifier')[
            ['accuracy', 'f1_weighted', 'f1_ps', 'precision_ps', 'recall_ps']
        ].agg(['mean', 'std'])
        print(classifier_stats.to_string())
        
        # Create comparison visualizations (separate academic figures)
        print(f"\n Creating performance comparison charts...")
        
        _original_rc = {k: plt.rcParams.get(k) for k in [
            'font.family', 'font.serif', 'axes.titlesize', 'axes.labelsize',
            'xtick.labelsize', 'ytick.labelsize']}
        plt.rcParams.update({
            'font.family': 'serif',
            'font.serif': ['Times New Roman', 'DejaVu Serif'],
            'axes.titlesize': 18, 'axes.labelsize': 16,
            'xtick.labelsize': 13, 'ytick.labelsize': 13,
        })
        
        # 1. F1 (PS) by Classifier
        classifier_f1 = results_df.groupby('classifier')['f1_ps'].mean().sort_values(ascending=False)
        fig1, ax1 = plt.subplots(figsize=(8, 6))
        ax1.barh(classifier_f1.index, classifier_f1.values, color='steelblue', alpha=0.85, edgecolor='white')
        ax1.set_xlabel('F1 Score (Process Safety)', fontsize=16)
        ax1.set_title('Average F1 Score by Classifier — Iteration 4', fontsize=18, fontweight='bold', pad=15)
        ax1.grid(axis='x', alpha=0.3)
        for i, v in enumerate(classifier_f1.values):
            ax1.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=12)
        fig1.tight_layout()
        fig1.savefig(os.path.join(RESULTS_DIR, 'comparison_f1ps_by_classifier.png'), dpi=300, bbox_inches='tight')
        fig1.savefig(os.path.join(RESULTS_DIR, 'comparison_f1ps_by_classifier.pdf'), bbox_inches='tight')
        plt.show(); plt.close(fig1)
        
        # 2. F1 (PS) by Model
        model_f1 = results_df.groupby('model')['f1_ps'].mean().sort_values(ascending=False)
        fig2, ax2 = plt.subplots(figsize=(8, 6))
        ax2.barh(model_f1.index, model_f1.values, color='coral', alpha=0.85, edgecolor='white')
        ax2.set_xlabel('F1 Score (Process Safety)', fontsize=16)
        ax2.set_title('Average F1 Score by Model — Iteration 4', fontsize=18, fontweight='bold', pad=15)
        ax2.grid(axis='x', alpha=0.3)
        for i, v in enumerate(model_f1.values):
            ax2.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=12)
        fig2.tight_layout()
        fig2.savefig(os.path.join(RESULTS_DIR, 'comparison_f1ps_by_model.png'), dpi=300, bbox_inches='tight')
        fig2.savefig(os.path.join(RESULTS_DIR, 'comparison_f1ps_by_model.pdf'), bbox_inches='tight')
        plt.show(); plt.close(fig2)
        
        # 3. Accuracy vs F1 (PS) scatter
        fig3, ax3 = plt.subplots(figsize=(8, 6))
        for classifier in results_df['classifier'].unique():
            subset = results_df[results_df['classifier'] == classifier]
            ax3.scatter(subset['accuracy'], subset['f1_ps'], label=classifier, alpha=0.7, s=100)
        ax3.set_xlabel('Accuracy', fontsize=16)
        ax3.set_ylabel('F1 Score (Process Safety)', fontsize=16)
        ax3.set_title('Accuracy vs F1 (PS) by Classifier — Iteration 4', fontsize=18, fontweight='bold', pad=15)
        ax3.legend(fontsize=11)
        ax3.grid(alpha=0.3)
        fig3.tight_layout()
        fig3.savefig(os.path.join(RESULTS_DIR, 'comparison_accuracy_vs_f1ps.png'), dpi=300, bbox_inches='tight')
        fig3.savefig(os.path.join(RESULTS_DIR, 'comparison_accuracy_vs_f1ps.pdf'), bbox_inches='tight')
        plt.show(); plt.close(fig3)
        
        # 4. Precision vs Recall for PS
        fig4, ax4 = plt.subplots(figsize=(8, 6))
        for classifier in results_df['classifier'].unique():
            subset = results_df[results_df['classifier'] == classifier]
            ax4.scatter(subset['recall_ps'], subset['precision_ps'], label=classifier, alpha=0.7, s=100)
        ax4.set_xlabel('Recall (Process Safety)', fontsize=16)
        ax4.set_ylabel('Precision (Process Safety)', fontsize=16)
        ax4.set_title('Precision vs Recall (PS) by Classifier — Iteration 4', fontsize=18, fontweight='bold', pad=15)
        ax4.legend(fontsize=11)
        ax4.grid(alpha=0.3)
        fig4.tight_layout()
        fig4.savefig(os.path.join(RESULTS_DIR, 'comparison_precision_vs_recall.png'), dpi=300, bbox_inches='tight')
        fig4.savefig(os.path.join(RESULTS_DIR, 'comparison_precision_vs_recall.pdf'), bbox_inches='tight')
        plt.show(); plt.close(fig4)
        
        # Restore rcParams
        plt.rcParams.update({k: v for k, v in _original_rc.items() if v is not None})
        
        print(f"[OK] All comparison charts saved (PNG + PDF)")

else:
    print(" No results to display.")

# Clear checkpoint if all completed
total_combinations = len(json_files) * len(MODELS)
if len(processed_items) == total_combinations:
    print(f"\n{'='*80}")
    print(" ALL COMBINATIONS PROCESSED!")
    print(f"{'='*80}")
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("  Checkpoint file cleared.")

total_time = (time.time() - start_time) / 60
print(f"\n Total execution time: {total_time:.2f} minutes")

print("\n" + "#"*80)
print(" ITERATION 4 PIPELINE COMPLETE")
print("#"*80)
print("\n Key Improvements Implemented:")
print("    Mean pooling for better embeddings")
print("    SMOTE for class balancing")
print("    Multiple classifier comparison")
print("    Hyperparameter tuning")
print("    Ensemble methods")
print("    Feature scaling")
print("    Enhanced text preprocessing")
print("    Comprehensive visualizations")
print("    ROC AUC and Precision-Recall curves")

/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



################################################################################
ITERATION 4: ENHANCED MULTI-MODEL PIPELINE
Advanced Techniques: SMOTE, Mean Pooling, Ensemble Methods, Hyperparameter Tuning
################################################################################

Using paths from Cell 2 (standardized):
   DATA_DIR:            /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k
   EMBEDDINGS_BASE_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Embeddings/_iteration_4
   RESULTS_DIR:         /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_4
   CHECKPOINT_FILE:     /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_4/checkpoint_iteration_4.json

 Configuration:
   Pooling Strategy: mean
   Use SMOTE: True
   Hyperparameter Tuning: True
   Ensemble Methods: True
   Models: ['bert-base-multilingual-uncased', 'xlm-roberta-base', 'paraphrase-multilingual-mpnet-base-v2', 

KeyboardInterrupt: 

# 3. Results Generation and Evaluation from Cached Embeddings

This cell provides an independent evaluation path that loads previously cached embeddings and retrains all classifiers with default hyperparameters. It generates thesis-quality figures (confusion matrices, metrics bar charts) for every model-dataset-classifier combination and compiles a comprehensive results CSV.

Use this cell when the main pipeline (Cell 6) has already generated embeddings and a quick re-evaluation of all classifiers is desired without re-running the full pipeline or hyperparameter search.

In [ ]:
# ============================================================
# RESULTS OF ITERATION 4 — Quick Evaluation Path
# (Trains classifiers with default hyperparameters from saved embeddings;
#  for GridSearchCV-tuned results, use the main pipeline in Cell 5.)
# ============================================================

import pandas as pd
import numpy as np
import os
import pickle
import glob
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, precision_recall_fscore_support,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import time

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False


print(f"Using paths from Cell 2:")
print(f"  EMBEDDINGS_BASE_DIR: {EMBEDDINGS_BASE_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
os.makedirs(RESULTS_DIR, exist_ok=True)

def calculate_detailed_metrics(y_test, y_pred):
    """Calculate comprehensive evaluation metrics"""
    cm = confusion_matrix(y_test, y_pred)
    TN, FP, FN, TP = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    total = TN + FP + FN + TP
    
    accuracy = (TP + TN) / total
    precision_ps = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall_ps = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1_ps = 2 * (precision_ps * recall_ps) / (precision_ps + recall_ps) if (precision_ps + recall_ps) > 0 else 0
    
    precision_nps = TN / (TN + FN) if (TN + FN) > 0 else 0
    recall_nps = TN / (TN + FP) if (TN + FP) > 0 else 0
    f1_nps = 2 * (precision_nps * recall_nps) / (precision_nps + recall_nps) if (precision_nps + recall_nps) > 0 else 0
    
    macro_f1 = (f1_ps + f1_nps) / 2
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
    fnr = FN / (FN + TP) if (FN + TP) > 0 else 0
    
    return {
        'confusion_matrix': cm,
        'TN': TN, 'FP': FP, 'FN': FN, 'TP': TP,
        'total': total, 'accuracy': accuracy,
        'precision_ps': precision_ps, 'recall_ps': recall_ps, 'f1_ps': f1_ps,
        'precision_nps': precision_nps, 'recall_nps': recall_nps, 'f1_nps': f1_nps,
        'f1_macro': macro_f1, 'fpr': fpr, 'fnr': fnr
    }

def print_detailed_results(dataset_name, model_key, classifier_name, metrics, cm):
    """Print formatted evaluation results with confusion matrix"""
    print(f"\n{'='*80}")
    print(f"RESULTS: {dataset_name.upper()} | {model_key} | {classifier_name}")
    print(f"{'='*80}")
    
    # Confusion Matrix Text
    print(f"\n{'CONFUSION MATRIX':^60}")
    print(f"{'-'*60}")
    print(f"{'':20} {'Pred: Non-PS':>18} {'Pred: PS':>18}")
    print(f"{'Actual: Non-PS':<20} {metrics['TN']:>18,} {metrics['FP']:>18,}")
    print(f"{'Actual: PS':<20} {metrics['FN']:>18,} {metrics['TP']:>18,}")
    print(f"{'-'*60}")
    
    # Performance Metrics
    print(f"\n{'PERFORMANCE METRICS':^60}")
    print(f"{'-'*60}")
    print(f"{'Metric':<20} {'Non-PS':>12} {'Process Safety':>15} {'Macro Avg':>12}")
    print(f"{'-'*60}")
    print(f"{'Precision':<20} {metrics['precision_nps']:>12.4f} {metrics['precision_ps']:>15.4f} {(metrics['precision_nps']+metrics['precision_ps'])/2:>12.4f}")
    print(f"{'Recall':<20} {metrics['recall_nps']:>12.4f} {metrics['recall_ps']:>15.4f} {(metrics['recall_nps']+metrics['recall_ps'])/2:>12.4f}")
    print(f"{'F1-Score':<20} {metrics['f1_nps']:>12.4f} {metrics['f1_ps']:>15.4f} {metrics['f1_macro']:>12.4f}")
    print(f"{'-'*60}")
    print(f"{'Accuracy':<20} {'':<27} {metrics['accuracy']:>12.4f}")
    print(f"{'FPR':<20} {'':<27} {metrics['fpr']:>12.4f}")
    print(f"{'FNR':<20} {'':<27} {metrics['fnr']:>12.4f}")
    print(f"{'='*60}\n")

def generate_results_from_embeddings(pkl_file, show_plots=True, save_plots=True):
    """Load embeddings and generate results with confusion matrices"""
    
    print(f"\n{'#'*80}")
    print(f"Loading: {os.path.basename(pkl_file)}")
    print(f"{'#'*80}")
    
    # Load embeddings
    with open(pkl_file, 'rb') as f:
        embeddings_data = pickle.load(f)
    
    X_train = embeddings_data['X_train_subset']
    y_train = embeddings_data['y_train_subset']
    X_test = embeddings_data['X_test_scaled']
    y_test = embeddings_data['y_test']
    metadata = embeddings_data['metadata']
    
    dataset_name = metadata['dataset']
    model_key = metadata['model_key']
    
    print(f"Dataset: {dataset_name.upper()}")
    print(f"Model: {model_key}")
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Test samples: {X_test.shape[0]}")
    print(f"Embedding dim: {X_train.shape[1]}")
    
    all_results = []
    
    # Define classifiers
    classifiers = {
        'SVM': LinearSVC(class_weight='balanced', max_iter=10000, dual=False, random_state=42),
        'Random_Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', 
                                                 random_state=42, n_jobs=-1, max_depth=20),
    }
    
    if XGBOOST_AVAILABLE:
        pos_weight = sum(y_train == 0) / sum(y_train == 1) if sum(y_train == 1) > 0 else 1
        classifiers['XGBoost'] = XGBClassifier(n_estimators=100, scale_pos_weight=pos_weight,
                                                random_state=42, n_jobs=-1, max_depth=6, learning_rate=0.1)
    
    if LIGHTGBM_AVAILABLE:
        classifiers['LightGBM'] = LGBMClassifier(n_estimators=100, class_weight='balanced',
                                                  random_state=42, n_jobs=-1, max_depth=6, 
                                                  learning_rate=0.1, verbose=-1)
    
    # Train and evaluate each classifier
    for clf_name, clf in classifiers.items():
        print(f"\n{'─'*60}")
        print(f"Training {clf_name}...")
        
        start_time = time.time()
        clf.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        y_pred = clf.predict(X_test)
        
        # Calculate metrics
        metrics = calculate_detailed_metrics(y_test, y_pred)
        cm = metrics['confusion_matrix']
        
        # Print detailed results
        print_detailed_results(dataset_name, model_key, clf_name, metrics, cm)
        
        # Print sklearn classification report
        print("Classification Report:")
        print(classification_report(y_test, y_pred, 
                                   target_names=['Non-Process Safety', 'Process Safety'],
                                   digits=4))
        
        # Create separate academic-quality figures
        safe_model = model_key.replace('/', '_')
        base_name = f"{dataset_name}_{safe_model}_{clf_name}"
        
        # --- Academic rcParams ---
        _original_rc = {k: plt.rcParams.get(k) for k in [
            'font.family', 'font.serif', 'axes.titlesize', 'axes.labelsize',
            'xtick.labelsize', 'ytick.labelsize']}
        plt.rcParams.update({
            'font.family': 'serif',
            'font.serif': ['Times New Roman', 'DejaVu Serif'],
            'axes.titlesize': 18, 'axes.labelsize': 16,
            'xtick.labelsize': 13, 'ytick.labelsize': 13,
        })
        
        # ---- 1. CONFUSION MATRIX (separate figure) ----
        fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Process Safety', 'Process Safety'],
            yticklabels=['Non-Process Safety', 'Process Safety'],
            ax=ax_cm, linewidths=0.8, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Count'},
            annot_kws={'size': 18, 'fontweight': 'bold'},
        )
        ax_cm.set_title(
            f'Confusion Matrix — {dataset_name.upper()}\n({model_key}, {clf_name})',
            fontsize=18, fontweight='bold', pad=15)
        ax_cm.set_ylabel('True Label', fontsize=16)
        ax_cm.set_xlabel('Predicted Label', fontsize=16)
        fig_cm.tight_layout()
        cm_path = os.path.join(RESULTS_DIR, f'confusion_matrix_{base_name}')
        fig_cm.savefig(cm_path + '.png', dpi=300, bbox_inches='tight')
        fig_cm.savefig(cm_path + '.pdf', bbox_inches='tight')
        print(f"[OK] Saved confusion matrix PNG + PDF: {os.path.basename(cm_path)}")
        plt.show()
        plt.close(fig_cm)
        
        # ---- 2. METRICS BAR CHART (separate figure) ----
        metrics_data = {
            'Accuracy': metrics['accuracy'],
            'F1 (PS)': metrics['f1_ps'],
            'Precision (PS)': metrics['precision_ps'],
            'Recall (PS)': metrics['recall_ps'],
            'F1 (Macro)': metrics['f1_macro'],
        }
        fig_bar, ax_bar = plt.subplots(figsize=(8, 6))
        ax_bar.barh(list(metrics_data.keys()), list(metrics_data.values()),
                    color='steelblue', alpha=0.85, edgecolor='white')
        ax_bar.set_xlim([0, 1.05])
        ax_bar.set_title(
            f'Performance Metrics — {dataset_name.upper()}\n({model_key}, {clf_name})',
            fontsize=18, fontweight='bold', pad=15)
        ax_bar.set_xlabel('Score', fontsize=16)
        ax_bar.grid(axis='x', alpha=0.3)
        for i, (k, v) in enumerate(metrics_data.items()):
            ax_bar.text(v + 0.015, i, f'{v:.4f}', va='center', fontsize=12)
        fig_bar.tight_layout()
        bar_path = os.path.join(RESULTS_DIR, f'metrics_bar_{base_name}')
        fig_bar.savefig(bar_path + '.png', dpi=300, bbox_inches='tight')
        fig_bar.savefig(bar_path + '.pdf', bbox_inches='tight')
        print(f"[OK] Saved metrics bar PNG + PDF: {os.path.basename(bar_path)}")
        plt.show()
        plt.close(fig_bar)
        
        # Restore rcParams
        plt.rcParams.update({k: v for k, v in _original_rc.items() if v is not None})
        
        # Get probability scores if available
        try:
            if hasattr(clf, 'predict_proba'):
                y_proba = clf.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_proba)
            elif hasattr(clf, 'decision_function'):
                y_proba = clf.decision_function(X_test)
                roc_auc = roc_auc_score(y_test, y_proba)
            else:
                y_proba = None
                roc_auc = None
        except:
            y_proba = None
            roc_auc = None
        
        # ---- 3. ROC CURVE (separate figure, if probabilities available) ----
        if y_proba is not None and roc_auc is not None:
            _original_rc2 = {k: plt.rcParams.get(k) for k in [
                'font.family', 'font.serif', 'axes.titlesize', 'axes.labelsize',
                'xtick.labelsize', 'ytick.labelsize']}
            plt.rcParams.update({
                'font.family': 'serif',
                'font.serif': ['Times New Roman', 'DejaVu Serif'],
                'axes.titlesize': 18, 'axes.labelsize': 16,
                'xtick.labelsize': 13, 'ytick.labelsize': 13,
            })
            fpr_vals, tpr_vals, _ = roc_curve(y_test, y_proba)
            fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
            ax_roc.plot(fpr_vals, tpr_vals, linewidth=2.2,
                        label=f'ROC (AUC = {roc_auc:.3f})')
            ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
            ax_roc.set_xlabel('False Positive Rate', fontsize=16)
            ax_roc.set_ylabel('True Positive Rate', fontsize=16)
            ax_roc.set_title(
                f'ROC Curve — {dataset_name.upper()}\n({model_key}, {clf_name})',
                fontsize=18, fontweight='bold', pad=15)
            ax_roc.legend(fontsize=13, loc='lower right')
            ax_roc.grid(alpha=0.3)
            fig_roc.tight_layout()
            roc_path = os.path.join(RESULTS_DIR, f'roc_curve_{base_name}')
            fig_roc.savefig(roc_path + '.png', dpi=300, bbox_inches='tight')
            fig_roc.savefig(roc_path + '.pdf', bbox_inches='tight')
            print(f"[OK] Saved ROC curve PNG + PDF: {os.path.basename(roc_path)}")
            plt.show()
            plt.close(fig_roc)
            
            # ---- 4. PRECISION-RECALL CURVE (separate figure) ----
            pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_proba)
            avg_prec = average_precision_score(y_test, y_proba)
            fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
            ax_pr.plot(pr_recall, pr_precision, linewidth=2.2,
                       label=f'AP = {avg_prec:.3f}')
            ax_pr.set_xlabel('Recall', fontsize=16)
            ax_pr.set_ylabel('Precision', fontsize=16)
            ax_pr.set_title(
                f'Precision-Recall Curve — {dataset_name.upper()}\n({model_key}, {clf_name})',
                fontsize=18, fontweight='bold', pad=15)
            ax_pr.legend(fontsize=13, loc='upper right')
            ax_pr.grid(alpha=0.3)
            fig_pr.tight_layout()
            pr_path = os.path.join(RESULTS_DIR, f'precision_recall_{base_name}')
            fig_pr.savefig(pr_path + '.png', dpi=300, bbox_inches='tight')
            fig_pr.savefig(pr_path + '.pdf', bbox_inches='tight')
            print(f"[OK] Saved PR curve PNG + PDF: {os.path.basename(pr_path)}")
            plt.show()
            plt.close(fig_pr)
            plt.rcParams.update({k: v for k, v in _original_rc2.items() if v is not None})
        
        # Store results
        all_results.append({
            'dataset': dataset_name,
            'model': model_key,
            'classifier': clf_name,
            'accuracy': metrics['accuracy'],
            'f1_ps': metrics['f1_ps'],
            'f1_nps': metrics['f1_nps'],
            'f1_macro': metrics['f1_macro'],
            'precision_ps': metrics['precision_ps'],
            'recall_ps': metrics['recall_ps'],
            'precision_nps': metrics['precision_nps'],
            'recall_nps': metrics['recall_nps'],
            'TN': metrics['TN'],
            'FP': metrics['FP'],
            'FN': metrics['FN'],
            'TP': metrics['TP'],
            'fpr': metrics['fpr'],
            'fnr': metrics['fnr'],
            'roc_auc': roc_auc,
            'training_time': training_time,
        })
    
    return all_results

def generate_all_results(show_plots=True, save_plots=True):
    """Generate results for all saved embeddings"""
    
    # Find all embedding files
    pkl_files = glob.glob(os.path.join(EMBEDDINGS_BASE_DIR, '**', '*.pkl'), recursive=True)
    
    print(f"\n{'#'*80}")
    print(f"GENERATING RESULTS FROM SAVED EMBEDDINGS")
    print(f"{'#'*80}")
    print(f"Found {len(pkl_files)} embedding files")
    
    if len(pkl_files) == 0:
        print(" No embedding files found! Run the main pipeline first.")
        return None
    
    all_results = []
    
    for pkl_file in pkl_files:
        try:
            results = generate_results_from_embeddings(pkl_file, show_plots, save_plots)
            all_results.extend(results)
        except Exception as e:
            print(f" Error processing {pkl_file}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    if all_results:
        # Create summary DataFrame
        results_df = pd.DataFrame(all_results)
        
        # Save to CSV
        summary_file = os.path.join(RESULTS_DIR, 'iteration_4_comprehensive_results.csv')
        results_df.to_csv(summary_file, index=False)
        print(f"\n Results saved to: {summary_file}")
        
        # Print summary tables
        print(f"\n\n{'#'*100}")
        print(f"{'ITERATION 4 - COMPREHENSIVE SUMMARY':^100}")
        print(f"{'#'*100}\n")
        
        # Confusion Matrix Summary
        print(f"{'='*100}")
        print(f"CONFUSION MATRIX SUMMARY BY MODEL-DATASET-CLASSIFIER")
        print(f"{'='*100}")
        print(f"{'Dataset':<12} {'Model':<35} {'Classifier':<15} {'TN':>8} {'FP':>8} {'FN':>8} {'TP':>8} {'Acc':>8} {'F1_PS':>8}")
        print(f"{'-'*100}")
        for _, row in results_df.iterrows():
            print(f"{row['dataset']:<12} {row['model'][:33]:<35} {row['classifier']:<15} {int(row['TN']):>8} {int(row['FP']):>8} {int(row['FN']):>8} {int(row['TP']):>8} {row['accuracy']:>8.4f} {row['f1_ps']:>8.4f}")
        print(f"{'='*100}\n")
        
        # Best performers
        print(f"\n{'='*80}")
        print(f"TOP 5 BEST PERFORMERS (by Process Safety F1)")
        print(f"{'='*80}")
        top5 = results_df.nlargest(5, 'f1_ps')
        for i, (_, row) in enumerate(top5.iterrows(), 1):
            print(f"\n{i}. {row['dataset'].upper()} | {row['model']} | {row['classifier']}")
            print(f"   F1 (PS): {row['f1_ps']:.4f} | Precision: {row['precision_ps']:.4f} | Recall: {row['recall_ps']:.4f}")
            print(f"   Accuracy: {row['accuracy']:.4f} | FNR: {row['fnr']:.4f}")
        
        # By classifier
        print(f"\n\n{'='*80}")
        print(f"AVERAGE PERFORMANCE BY CLASSIFIER")
        print(f"{'='*80}")
        clf_stats = results_df.groupby('classifier')[['accuracy', 'f1_ps', 'precision_ps', 'recall_ps', 'fnr']].mean()
        print(clf_stats.round(4).to_string())
        
        # By model
        print(f"\n\n{'='*80}")
        print(f"AVERAGE PERFORMANCE BY MODEL")
        print(f"{'='*80}")
        model_stats = results_df.groupby('model')[['accuracy', 'f1_ps', 'precision_ps', 'recall_ps', 'fnr']].mean()
        print(model_stats.round(4).to_string())
        
        return results_df
    
    return None

# ============================================================
# RUN THE RESULTS GENERATOR
# ============================================================

# Generate results for all saved embeddings
results_df = generate_all_results(show_plots=True, save_plots=True)

if results_df is not None:
    print(f"\n Successfully generated results for {len(results_df)} model-dataset-classifier combinations")
else:
    print("\n No results generated - check that embeddings exist")

# 4. Iteration 4 Summary and Export

This cell reads the comprehensive results CSV generated by the evaluation pipeline, identifies the best-performing model-dataset-classifier combination by Process Safety F1-score, and exports a structured `iteration_4_summary.json` file. The summary includes per-combination confusion matrices, performance metrics, and a gap analysis relative to the RQ1 target ($\text{macro-F1} \geq 0.85$).

In [ ]:
# =============================================================================
# ITERATION 4 SUMMARY — Save iteration_4_summary.json
# =============================================================================
# Consistent with Iteration 0's summary output format.

import json
import pandas as pd
import os

_results_csv = os.path.join(RESULTS_DIR, 'iteration_4_comprehensive_results.csv')

if os.path.exists(_results_csv):
    _results_df = pd.read_csv(_results_csv)
    _best = _results_df.loc[_results_df['f1_ps'].idxmax()]
    _RQ1_TARGET = 0.85

    _summary = {
        'iteration': 4,
        'model': '10+ advanced embedding models + 7 classifiers (GridSearchCV, ensemble) + SMOTE',
        'best_dataset':  str(_best['dataset']),
        'best_model':    str(_best['model']),
        'best_classifier': str(_best['classifier']),
        'datasets': {},
        'gap_analysis': {}
    }

    for _, row in _results_df.iterrows():
        key = f"{row['dataset']}__{row['model']}__{row['classifier']}"
        _summary['datasets'][key] = {
            'total_samples': int(row.get('TN', 0)) + int(row.get('FP', 0)) + int(row.get('FN', 0)) + int(row.get('TP', 0)),
            'results': {
                'accuracy':     round(float(row['accuracy']), 4),
                'f1_ps':        round(float(row['f1_ps']), 4),
                'precision_ps': round(float(row['precision_ps']), 4),
                'recall_ps':    round(float(row['recall_ps']), 4),
            },
            'confusion_matrix': {
                'tn': int(row['TN']), 'fp': int(row['FP']),
                'fn': int(row['FN']), 'tp': int(row['TP']),
            }
        }
        _summary['gap_analysis'][key] = {
            'current_f1_ps': round(float(row['f1_ps']), 4),
            'rq1_target': _RQ1_TARGET,
            'gap': round(float(_RQ1_TARGET - row['f1_ps']), 4),
            'improvement_over_baseline': round(float(row['f1_ps']) - 0.7825, 4)
        }

    _out_path = os.path.join(RESULTS_DIR, 'iteration_4_summary.json')
    with open(_out_path, 'w', encoding='utf-8') as _f:
        json.dump(_summary, _f, indent=2, ensure_ascii=False)
    print(f"[OK] Saved: {_out_path}")
    print(f"Best: {_best['dataset']} | {_best['model']} | {_best['classifier']} | F1_PS={_best['f1_ps']:.4f}")
else:
    print(f"[INFO] Results CSV not found yet: {_results_csv}")
    print("       Run the pipeline first, then re-run this cell.")